# 04. Text Generation Decoding and Structured Grammar Sampling

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** Holtzman et al. (2019, *Curious Case of Neural Text Degeneration*), Keskar et al. (2019, *CTRL*), Willard & Louf (2023, *Outlines*), Nguyen et al. (2024, *Min-P Sampling*).
- **Purpose:** Implement and verify modern autoregressive decoding pipelines combining repetition penalties, pre-softmax FSM grammar masking for 100% valid JSON/code schemas, temperature scaling, and dynamic Min-P truncation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [AOT Inductor and ExecuTorch Edge Concepts](03_aot_inductor_and_executorch_edge_concepts.ipynb) $\to$ `🔗 Downstream Syntheses:` [Production Serving Dynamic Batcher Pd Disaggregation and Streaming](05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not apply Top-$P$ or Min-$P$ filtering before temperature scaling, as probability distribution tail shapes are distorted. Avoid post-hoc rejection sampling or regex stripping for structured outputs, which wastes up to $90\%$ of forward-pass inference tokens.

## Mathematical Foundations & Governing Equations

$$
\tilde{z}_i = \begin{cases} z_i / \alpha_{\text{rep}}, & z_i > 0, i \in \mathcal{H} \\ z_i \cdot \alpha_{\text{rep}}, & z_i \le 0, i \in \mathcal{H} \end{cases}, \quad z^{\text{mask}}_i = \begin{cases} \tilde{z}_i, & i \in \mathcal{A}(s) \\ -10^4, & i \notin \mathcal{A}(s) \end{cases}, \quad p_i = \text{softmax}\left( \frac{z^{\text{mask}}_i}{T} \right)
$$
$$
\bar{p}_i = p_i \cdot \mathbb{I}\left( p_i \ge p_{\min} \cdot \max_j p_j \right), \quad p^{\text{final}}_i = \frac{\bar{p}_i}{\sum_j \bar{p}_j}
$$

### Asymptotic Complexity & Operational Guarantees
- **FSM Masking:** $O(1)$ precomputed bitmask lookup per state transition vs $O(V)$ grammar parsing overhead.
- **Min-$P$ Truncation:** $O(V)$ dynamically adaptive tail pruning preserving semantic diversity without tail degeneration.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
MASK_VALUE = -10000.0

def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def apply_repetition_penalty(logits: torch.Tensor, history: torch.Tensor, penalty: float = 1.25) -> torch.Tensor:
    out = logits.clone()
    for b in range(logits.shape[0]):
        for tok in history[b]:
            idx = int(tok.item())
            val = out[b, idx]
            out[b, idx] = val / penalty if val > 0 else val * penalty
    return out

def apply_grammar_mask(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return torch.where(mask > 0.5, logits, torch.full_like(logits, MASK_VALUE))

def apply_temperature(logits: torch.Tensor, temperature: float = 0.8) -> torch.Tensor:
    return logits / max(temperature, 1e-5)

def apply_top_k(logits: torch.Tensor, k: int = 4) -> torch.Tensor:
    val, _ = torch.topk(logits, k=k, dim=-1)
    return torch.where(logits >= val[:, -1:], logits, torch.full_like(logits, MASK_VALUE))

def apply_min_p(probs: torch.Tensor, min_p: float = 0.1) -> torch.Tensor:
    p_max, _ = torch.max(probs, dim=-1, keepdim=True)
    filtered = torch.where(probs >= p_max * min_p, probs, torch.zeros_like(probs))
    return filtered / torch.sum(filtered, dim=-1, keepdim=True).clamp_min(1e-12)

def decoding_pipeline(logits, history, mask, temp=0.8, rep=1.25, k=4, min_p=0.1):
    x = apply_repetition_penalty(logits, history, rep)
    x = apply_grammar_mask(x, mask)
    x = apply_temperature(x, temp)
    x = apply_top_k(x, k)
    probs = apply_min_p(F.softmax(x, dim=-1), min_p)
    return probs, torch.argmax(probs, dim=-1).to(torch.float32)

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

rep = apply_repetition_penalty(weights["logits"], weights["history"], 1.25)
assert float(torch.max(torch.abs(rep - weights["rep_penalized"]))) < 1e-5

masked = apply_grammar_mask(rep, weights["grammar_mask"])
assert float(torch.max(torch.abs(masked - weights["masked_logits"]))) < 1e-5

temp = apply_temperature(masked, 0.8)
assert float(torch.max(torch.abs(temp - weights["temp_logits"]))) < 1e-5

topk = apply_top_k(temp, 4)
assert float(torch.max(torch.abs(topk - weights["top_k_logits"]))) < 1e-5

probs, greedy = decoding_pipeline(weights["logits"], weights["history"], weights["grammar_mask"])
assert float(torch.max(torch.abs(probs - weights["probs"]))) < 1e-5
assert float(torch.max(torch.abs(greedy - weights["greedy_tokens"]))) < 1e-5
print("All Text Generation Decoding & Structured Grammar Sampling parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: apply_repetition_penalty(weights["logits"], weights["history"], 1.25), iters=50, name="Repetition Penalty"),
    benchmark_fn(lambda: decoding_pipeline(weights["logits"], weights["history"], weights["grammar_mask"]), iters=50, name="Full Decoding Pipeline"),
]
print(format_benchmark_table(benchmarks))

# SVG Visualization of Decoding Pipeline
svg = """<svg width="480" height="80" xmlns="http://www.w3.org/2000/svg">
  <rect width="480" height="80" fill="#1e1e2e" rx="8"/>
  <rect x="10" y="20" width="80" height="40" fill="#313244" rx="4" stroke="#89b4fa" stroke-width="1.5"/>
  <text x="50" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Logits</text>
  <path d="M95 40 L115 40" stroke="#89b4fa" stroke-width="2" marker-end="url(#arrow)"/>
  <rect x="120" y="20" width="90" height="40" fill="#313244" rx="4" stroke="#a6e3a1" stroke-width="1.5"/>
  <text x="165" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Rep Penalty</text>
  <path d="M215 40 L235 40" stroke="#a6e3a1" stroke-width="2"/>
  <rect x="240" y="20" width="100" height="40" fill="#313244" rx="4" stroke="#f38ba8" stroke-width="1.5"/>
  <text x="290" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">FSM Mask</text>
  <path d="M345 40 L365 40" stroke="#f38ba8" stroke-width="2"/>
  <rect x="370" y="20" width="95" height="40" fill="#313244" rx="4" stroke="#fab387" stroke-width="1.5"/>
  <text x="417" y="44" fill="#cdd6f4" font-size="11" text-anchor="middle">Min-P Sample</text>
</svg>"""
from IPython.display import display, HTML
display(HTML(svg))

In [ ]:
# Pathological Misuse: Unconstrained Stochastic Sampling vs FSM Grammar Masking
torch.manual_seed(123)
# Simulating unconstrained sampling where token 15 violates JSON schema syntax
raw_logits = torch.randn(1, 16)
raw_logits[0, 15] = 5.0  # high confidence illegal token

# Naive unconstrained greedy decode selects illegal token 15:
unconstrained_tok = int(torch.argmax(raw_logits, dim=-1).item())

# Grammar-constrained decode masks out token 15 before selection:
fsm_mask = torch.ones(1, 16)
fsm_mask[0, 15] = 0.0  # token 15 disallowed by grammar
constrained_logits = apply_grammar_mask(raw_logits, fsm_mask)
constrained_tok = int(torch.argmax(constrained_logits, dim=-1).item())

print(f"Unconstrained Token Selected:  {unconstrained_tok} (Illegal token! Syntactic error)")
print(f"FSM Constrained Token:        {constrained_tok} (100% Valid schema transition)")
assert unconstrained_tok == 15 and constrained_tok != 15, "Failure pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Pre-Softmax Masking:** Precomputing FSM transition masks guarantees 100% syntactically valid JSON/SELFIES outputs on the first pass without rejection overhead.
2. **Min-$P$ Dynamic Tail Pruning:** Min-$P$ scales the truncation threshold with top-token confidence, eliminating tail hallucinations under temperature scaling.
3. **Pipeline Order Invariance:** Repetition penalty must precede grammar masking and temperature scaling to preserve normalized probability calibrations.

---
### Cross-Domain Generalisation Matrix
| Domain | Constrained Generation Analogue | Guarantee |
| :--- | :--- | :--- |
| **Compiler & Code Generation** | LR(1) / PEG Parser state bitmasking | Guarantees zero abstract syntax tree parse errors on generation |
| **Chemoinformatics (SELFIES)** | Chemical valence bond graph grammar masks | Guarantees 100% valid chemical valency and molecular synthesizability |
| **Robotic Motion Planning** | Kinematic reachability convex hull masking | Prevents actuator over-extension and collisions in unconstrained control |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_text_generation_decoding_and_structured_grammar_sampling.md](../../markdown/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.md)
- **PyTorch 2.x Module:** [04_text_generation_decoding_and_structured_grammar_sampling.py](../../pytorch2/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.py)
- **Functional JAX Module:** [04_text_generation_decoding_and_structured_grammar_sampling.py](../../jax/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.py)
- **Native Rust Candle Track:** [04_text_generation_decoding_and_structured_grammar_sampling.rs](../../candle/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.rs)
- **Hardware Kernel (CPU / WGSL):** [04_text_generation_decoding_and_structured_grammar_sampling.rs](../../kernels/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.rs)
- **Golden SafeTensors:** [04_text_generation_decoding_and_structured_grammar_sampling.safetensors](../../weights/17_deployment_low_level_and_capstones/04_text_generation_decoding_and_structured_grammar_sampling.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. AOT Inductor and ExecuTorch Edge Concepts](03_aot_inductor_and_executorch_edge_concepts.ipynb) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 05. Production Serving Dynamic Batcher Pd Disaggregation and Streaming](05_production_serving_dynamic_batcher_pd_disaggregation_and_streaming.ipynb) &rarr;